# 07 — Inference Testing

Load the trained baseline HMER checkpoint and run greedy decoding on a processed CROHME image. 

This notebook expects `src/inference.py`, `data/processed/crohme2019/metadata/vocab.json`, the processed manifests/images, and a saved checkpoint under `outputs/models/`.

**Important:** if your training notebook saves a checkpoint under a different filename or format, set `CHECKPOINT_PATH` below 
and ensure it contains weights compatible with `BaselineHMER`.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Math

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.inference import HMERInference

PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'crohme2019'
MANIFEST_ROOT = PROCESSED_ROOT / 'manifests'
VOCAB_PATH = PROCESSED_ROOT / 'metadata' / 'vocab.json'
CHECKPOINT_PATH = None  # Set explicitly if auto-discovery does not select your trained checkpoint.

assert VOCAB_PATH.is_file(), f'Missing vocabulary: {VOCAB_PATH}'
assert (MANIFEST_ROOT / 'test.jsonl').is_file(), 'Missing test manifest. Run preprocessing/validation first.'
print('Project root:', PROJECT_ROOT)
print('Vocabulary:', VOCAB_PATH)

In [ ]:
# Initialize inference once; this loads the checkpoint and vocabulary.
predictor = HMERInference(
    checkpoint_path=CHECKPOINT_PATH,
    vocab_path=VOCAB_PATH,
    project_root=PROJECT_ROOT,
)
print('Checkpoint:', predictor.checkpoint_path)
print('Device:', predictor.device)
print('Vocabulary size:', len(predictor.token_to_id))

In [ ]:
# Load test manifest and select the first record whose processed image exists.
test_records = []
with (MANIFEST_ROOT / 'test.jsonl').open('r', encoding='utf-8') as file:
    for line in file:
        if line.strip():
            test_records.append(json.loads(line))

def resolve_test_image(record):
    image_value = record.get('image')
    if image_value:
        candidate = Path(image_value)
        if not candidate.is_absolute():
            candidate = PROCESSED_ROOT / candidate
        if candidate.is_file():
            return candidate
    split = record.get('split', 'test')
    record_id = record.get('id')
    if record_id:
        candidate = PROCESSED_ROOT / 'images' / split / f'{record_id}.png'
        if candidate.is_file():
            return candidate
    return None

sample_record = None
sample_image_path = None
for record in test_records:
    candidate = resolve_test_image(record)
    if candidate is not None:
        sample_record, sample_image_path = record, candidate
        break

if sample_record is None:
    raise FileNotFoundError('No test image could be resolved from test.jsonl. Check manifest image paths.')

print('Sample ID:', sample_record.get('id'))
print('Image:', sample_image_path)
print('Ground truth:', sample_record.get('latex', '<not provided>'))

In [ ]:
# Predict and inspect one expression.
prediction = predictor.predict(sample_image_path, max_length=200)
print('Predicted LaTeX:', prediction['latex'])
print('EOS reached:', prediction['terminated'])
print('Generated token count:', len(prediction['token_ids']))

with Image.open(sample_image_path) as image:
    plt.figure(figsize=(12, 3))
    plt.imshow(image.convert('L'), cmap='gray')
    plt.axis('off')
    plt.title(f"Test sample: {sample_record.get('id', '')}")
    plt.show()

ground_truth = sample_record.get('latex', '')
print('Ground-truth LaTeX:', ground_truth)
try:
    if ground_truth:
        print('Ground truth rendered:')
        display(Math(ground_truth.strip('$')))
    print('Prediction rendered:')
    display(Math(prediction['latex']))
except Exception as exc:
    print('MathJax/IPython rendering failed; inspect the raw LaTeX above.', exc)

## Run inference on a custom image

Set `CUSTOM_IMAGE_PATH` to an image file containing one handwritten mathematical expression. This uses the same preprocessing as the baseline training dataset.

In [ ]:
CUSTOM_IMAGE_PATH = None  # Example: PROJECT_ROOT / 'data' / 'my_expression.png'

if CUSTOM_IMAGE_PATH is not None:
    custom_result = predictor.predict(CUSTOM_IMAGE_PATH, max_length=200)
    print('Predicted LaTeX:', custom_result['latex'])
    print('EOS reached:', custom_result['terminated'])
    display(Math(custom_result['latex']))
else:
    print('Set CUSTOM_IMAGE_PATH to run inference on your own image.')

## Interpretation and limitations

- This notebook runs **greedy decoding**: the highest-probability next token is selected at each step.
- `EOS reached: False` means generation stopped at `max_length`, not at the model's end-of-sequence token.
- A prediction is only meaningful after the model has been trained and the correct checkpoint is loaded.
- The current baseline compresses the full image into one feature vector and does not use attention. Complex spatial structures may therefore be difficult to recognize.
- This notebook is a smoke test, not a substitute for evaluation on the untouched test set.